In [ ]:

!pip install git+https://github.com/Lightning-AI/litgpt.git

In [ ]:
# run this 2x make sure you see /content/hf_cache directory
# first time it installs and restarts session without completing this cell

!pip install -U datasets fsspec

import os
os.environ["HF_DATASETS_CACHE"] = "/content/hf_cache"

from datasets import load_dataset

dataset = load_dataset("mhenrichsen/alpaca_2k_test")
print(dataset)

In [ ]:
/content/drive/MyDrive/huggingface_model_downloads/meta-llama/models--meta-llama--Meta-Llama-3-8B/snapshots/8cde5ca8380496c9a6cc7ef3a8b46a0372a1d920

In [ ]:
import torch
import litgpt
from litgpt.lora import GPT, merge_lora_weights
from litgpt.data import Alpaca2k # this isnt verified
import lightning as L




class LitLLM(L.LightningModule):
    def __init__(self):
        super().__init__()
        self.model = GPT.from_name(
            name="Llama-3.1-8B",
            lora_r=32,
            lora_alpha=16,
            lora_dropout=0.05,
            lora_query=True,
            lora_key=False,
            lora_value=True,
        )
        litgpt.lora.mark_only_lora_as_trainable(self.model)

    def on_train_start(self):
        state_dict = torch.load("/content/drive/MyDrive/huggingface_model_downloads/meta-llama/models--meta-llama--Meta-Llama-3-8B/snapshots/8cde5ca8380496c9a6cc7ef3a8b46a0372a1d920/pytorch_model.bin", mmap=True)
        self.model.load_state_dict(state_dict, strict=False)

    def training_step(self, batch):
        input_ids, targets = batch["input_ids"], batch["labels"]
        logits = self.model(input_ids)
        loss = litgpt.utils.chunked_cross_entropy(logits[..., :-1, :], targets[..., 1:])
        self.log("train_loss", loss, prog_bar=True)
        return loss

    def configure_optimizers(self):
        warmup_steps = 10
        optimizer = torch.optim.AdamW(self.model.parameters(), lr=0.0002, weight_decay=0.0, betas=(0.9, 0.95))
        scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lambda step: step / warmup_steps)
        return [optimizer], [scheduler]


if __name__ == "__main__":
    data = Alpaca2k()
    tokenizer = litgpt.Tokenizer("/content/drive/MyDrive/huggingface_model_downloads/meta-llama/models--meta-llama--Meta-Llama-3-8B/snapshots/8cde5ca8380496c9a6cc7ef3a8b46a0372a1d920")
    data.connect(tokenizer, batch_size=1, max_seq_length=512)

    trainer = L.Trainer(
        devices=1,
        max_epochs=2,
        accumulate_grad_batches=8,
        precision="bf16-true",
    )
    with trainer.init_module(empty_init=True):
        model = LitLLM()
    print(f"data:{data}")
    print(f"model:{model}")

    trainer.fit(model, data)

    # Save final checkpoint
    merge_lora_weights(model.model)
    #stored in /content/drive/MyDrive/meta-llama/Meta-Llama-3-8B/finetuned.ckpt
    trainer.save_checkpoint("meta-llama/Meta-Llama-3-8B/finetuned.ckpt", weights_only=True)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content/drive/MyDrive

In [ ]:
!pip install transformers accelerate

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
# device memory use cpu not cuda
model = AutoModelForCausalLM.from_pretrained("meta-llama/Meta-Llama-3-8B", device_map="auto")
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Meta-Llama-3-8B")


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
# copied from /root/.cache to /content/drive/MyDrive....

#model = AutoModelForCausalLM.from_pretrained("/content/drive/MyDrive/huggingface_model_downloads/meta-llama/models--meta-llama--Meta-Llama-3-8B/snapshots/8cde5ca8380496c9a6cc7ef3a8b46a0372a1d920", device_map="auto")
#tokenizer = AutoTokenizer.from_pretrained("/content/drive/MyDrive/huggingface_model_downloads/meta-llama/models--meta-llama--Meta-Llama-3-8B/snapshots/8cde5ca8380496c9a6cc7ef3a8b46a0372a1d920")

In [ ]:
from accelerate import init_empty_weights, infer_auto_device_map, load_checkpoint_and_dispatch
from transformers import AutoConfig, AutoModelForCausalLM

model_name = "/content/drive/MyDrive/huggingface_model_downloads/meta-llama/models--meta-llama--Meta-Llama-3-8B/snapshots/8cde5ca8380496c9a6cc7ef3a8b46a0372a1d920"

config = AutoConfig.from_pretrained(model_name)

with init_empty_weights():
    model = AutoModelForCausalLM.from_config(config)

device_map = infer_auto_device_map(model, max_memory={0: "13GiB", "cpu": "30GiB"}, no_split_module_classes=["LlamaDecoderLayer"])

model = load_checkpoint_and_dispatch(
    model,
    checkpoint=model_name,
    device_map=device_map,
    offload_folder="./offload"
)

In [ ]:
!pip install safetensors

In [ ]:
import torch
from safetensors.torch import load_file
import os

# Path to the directory containing model shards
shard_dir = "/content/drive/MyDrive/huggingface_model_downloads/meta-llama/models--meta-llama--Meta-Llama-3-8B/snapshots/8cde5ca8380496c9a6cc7ef3a8b46a0372a1d920"  # 🔁 Replace with your actual path

# List all .safetensors shard files
safetensor_files = sorted([
    os.path.join(shard_dir, f) for f in os.listdir(shard_dir)
    if f.endswith(".safetensors")
])

print(f"Found {len(safetensor_files)} safetensor shards.")

# Merge all shard dictionaries
full_state_dict = {}
for file in safetensor_files:
    shard = load_file(file)
    full_state_dict.update(shard)

# Save the merged state_dict as pytorch_model.bin
torch.save(full_state_dict, os.path.join(shard_dir, "pytorch_model.bin"))
print("Saved combined model to pytorch_model.bin")